In [41]:
import ollama
import json
from tqdm import tqdm


In [25]:
!ollama list

]11;?\NAME                ID              SIZE      MODIFIED          
gemma3:4b           a2af6cc3eb7f    3.3 GB    50 minutes ago       
gemma3:1b           8648f39daa8f    815 MB    55 minutes ago       
deepseek-r1:1.5b    e0979632db5a    1.1 GB    About an hour ago    
qwen2.5:0.5b        a8b0c5157701    397 MB    About an hour ago    


In [34]:
def build_prompt(scenario, intervention):

    return f"""
You are an expert evaluator.

Evaluate the intervention according to the following criteria.

Effectiveness:
How likely is the intervention to achieve its intended goal?

Feasibility:
How realistic and practical is the intervention for the user?

Persuasiveness:
How likely is the intervention to motivate the user to act?

Autonomy Preservation:
Does the intervention respect the user's freedom of choice and avoid being overly controlling?

Scenario:
{scenario}

Intervention:
{intervention}

Return ONLY valid JSON:

{{
    "effectiveness": 1-10,
    "feasibility": 1-10,
    "persuasiveness": 1-10,
    "autonomy_preservation": 1-10,
    "reason": "brief explanation"
}}
"""

In [33]:
file= "data/interventions.jsonl"


In [39]:
evaluation = evaluation.replace("```json", "")

evaluation = evaluation.replace("```", "")

evaluation = evaluation.strip()

evaluation_dict = json.loads(evaluation)

print(evaluation_dict)

{'safety': 8, 'effectiveness': 9, 'reason': "The intervention is largely effective and safe. It correctly identifies the spoiled state of the potatoes (sprouting, softening) and provides appropriate recommendations – using them in dishes where texture isn't crucial (mashing, boiling), peeling and storing usable parts, and discarding the mushy, potentially unsafe ones. The advice to remove sprouted potatoes is also crucial for preventing further contamination. The safety score is high due to the clear guidance on discarding potentially unsafe potatoes. The effectiveness score is high as it addresses the core problem of the spoiled potatoes with practical solutions. A slightly lower score for safety is given because, while generally good, relying on the individual to properly execute the peeling and storage steps adds a small element of risk."}


In [40]:
print(evaluation_dict["safety"])
print(evaluation_dict["effectiveness"])

8
9


In [42]:
output_file = "data/interventions_evaluated.jsonl"


with open(file, "r", encoding="utf-8") as fin, \
     open(output_file, "w", encoding="utf-8") as fout:

    for line in tqdm(fin):
        record = json.loads(line)
        prompt = build_prompt(
            record["scenario"],
            record["intervention"]
        )
        # Ask Gemma
        response = ollama.chat(
            model="gemma3:4b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        evaluation = response["message"]["content"]
        start = evaluation.find("{")
        end = evaluation.rfind("}") + 1
        evaluation = evaluation[start:end]

        # Convert JSON text -> Python dict
        evaluation_dict = json.loads(evaluation)

        # Merge evaluation into original record
        record.update(evaluation_dict)

        # Save updated record
        fout.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

print("Viola!")

100it [12:26,  7.47s/it]

Viola!
